In [1]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt

/Users/ywo/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
db=wrds.Connection()

Enter your WRDS username [ywo]:ywhan
Enter your password:········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


author's data

In [3]:
df2 = pd.read_csv('/Users/ywo/Downloads/Industry Project/datashare (1)/datashare.csv')
df2['DATE']=pd.to_datetime(df2['DATE'], format='%Y%m%d')
# print(df2)

In [4]:
print(df2[(df2['DATE']<='1987-12-31')&(df2['permno']==10006)])

         permno       DATE       mvel1      beta    betasq     chmom  \
0         10006 1957-01-31   82249.000  1.122846  1.260784  0.047180   
1053      10006 1957-02-28   87544.000  1.100682  1.211500  0.030330   
2106      10006 1957-03-29   86308.500  1.117907  1.249717  0.134574   
3164      10006 1957-04-30   87897.000  1.110503  1.233218  0.253516   
4223      10006 1957-05-31   87191.000  1.105519  1.222172  0.206966   
...         ...        ...         ...       ...       ...       ...   
1031848   10006 1984-02-29  382565.625  0.975052  0.950726  0.321966   
1038049   10006 1984-03-30  400383.750  0.923933  0.853652  0.499693   
1044282   10006 1984-04-30  427635.000  0.939330  0.882342 -0.568023   
1050562   10006 1984-05-31  441260.625  0.943129  0.889492 -0.369969   
1056861   10006 1984-06-29  441260.625  0.923484  0.852823 -0.516111   

            dolvol   idiovol    indmom     mom1m  ...     stdcf   ms  \
0         9.569953  0.025742  0.046433  0.044843  ...       NaN

asset growth

In [ ]:
# author's data
df2_agr=df2[['DATE','permno','agr']]
df2_agr=df2_agr[df2_agr['agr'].notna()]
df2_agr=df2_agr[df2_agr['permno']==22840]
df2_agr.head(24)

In [9]:
# feature construction
sql_query = """
SELECT gvkey, datadate, fyear, at
FROM comp.funda
WHERE indfmt='INDL' 
  AND datafmt='STD' 
  AND popsrc='D' 
  AND consol='C'
  AND fyear BETWEEN 1956 AND 1988
"""

comp = db.raw_sql(sql_query)

link_query = """
SELECT gvkey, lpermno AS permno, linkdt, linkenddt
FROM crsp.ccmxpf_linktable
WHERE linktype IN ('LU', 'LC') 
  AND linkprim IN ('P', 'C')
"""
link = db.raw_sql(link_query)

In [10]:
link['permno'].astype(int)
# print(link)
agr=pd.merge(comp,link,on='gvkey',how='inner')
agr = agr[(agr['datadate'] >= agr['linkdt']) & ((agr['datadate'] <= agr['linkenddt']) | (agr['linkenddt'].isna()))]
# print(agr)
agr=agr[['datadate','fyear','permno','at']]

In [11]:
agr=agr.sort_values(['permno','fyear'])
agr = agr.drop_duplicates(subset=['permno', 'datadate'], keep='first')
agr=agr[agr['at'].notna()]
agr=agr.reset_index(drop=True)
agr['at_1']=agr.groupby('permno')['at'].shift(1)
agr['fyear_1']=agr.groupby('permno')['fyear'].shift(1)
agr.loc[agr['fyear']==agr['fyear_1']+1,'agr']=agr['at']/agr['at_1']-1
valid_agr = agr.loc[agr['agr'].notna(), 'agr'].astype(float)
from scipy.stats.mstats import winsorize
agr.loc[agr['agr'].notna(), 'agr'] = winsorize(valid_agr, limits=[0.01, 0.01]).data
agr=agr[agr['agr'].notna()]

agr['datadate']=pd.to_datetime(agr['datadate'])
agr['valid_from'] = (agr['datadate'] + pd.DateOffset(months=7)).dt.to_period('M')
agr=agr[['datadate','fyear','permno','agr','valid_from']]


In [ ]:
# print(agr[agr['permno']==22840])

In [12]:
df_feature=pd.read_csv('/Users/ywo/Downloads/Industry Project/feature construction.csv')
# df_feature

In [ ]:
df_feature

In [13]:
df_feature['date'] = pd.to_datetime(df_feature['date'], errors='coerce')
# agr['valid_from'] = pd.to_datetime(agr['valid_from'], errors='coerce')
df_feature['year_month'] = df_feature['date'].dt.to_period('M')
df_feature['permno'] = pd.to_numeric(df_feature['permno'], errors='coerce').astype(int)
agr['permno'] = pd.to_numeric(agr['permno'], errors='coerce').astype(int)
# df_feature = df_feature.dropna(subset=['date'])
# agr = agr.dropna(subset=['valid_from'])
# print(df_feature)
# print(agr)


In [14]:
df_final = pd.merge(
    df_feature, 
    agr, 
    left_on=['permno', 'year_month'], 
    right_on=['permno', 'valid_from'], 
    how='left'
)

df_final['agr'] = df_final.groupby('permno')['agr'].ffill(limit=11)
df_final=df_final.drop(columns=['year_month', 'valid_from','fyear','datadate'])

In [ ]:
df_final

beta

In [3]:
import pandas as pd
import numpy as np
import wrds
from pandas.tseries.offsets import MonthEnd

In [4]:

sql_query = """
    select a.permno, a.date, a.ret, b.rf, b.mktrf
    from crsp.dsf as a
    left join ff.factors_daily as b
    on a.date = b.date
    where a.date >= '1956-08-01' and a.date <= '1987-12-31'
"""
crsp = db.raw_sql(sql_query)

crsp = crsp.drop_duplicates(subset=['permno', 'date'], keep='first')
crsp = crsp.reset_index(drop=True)
crsp['permno'] = crsp['permno'].astype(int)
crsp['date'] = pd.to_datetime(crsp['date'])
crsp = crsp.sort_values(['permno', 'date'])

# Excess Return
crsp['exret'] = crsp['ret'] - crsp['rf']
crsp['exmkf']= crsp['mktrf'] - crsp['rf']


In [5]:
crsp['exret'] = crsp['exret'].astype('float64')
crsp['exmkf'] = crsp['exmkf'].astype('float64')

In [6]:

import gc
crsp['monthend_date'] = crsp['date'] + MonthEnd(0)

crsp['is_last_trade_day'] = crsp.groupby(['permno', 'monthend_date'])['date'].transform('max') == crsp['date']

def calculate_beta_numpy(df_group):


    y = df_group['exret'].values
    x = df_group['exmkf'].values
    

    betas = np.full(len(y), np.nan)
    
   
    # window=63, min_periods=21
    for i in range(20, len(y)):
        start = max(0, i - 62)
        y_window = y[start:i+1]
        x_window = x[start:i+1]
        
        
        valid_mask = ~np.isnan(y_window) & ~np.isnan(x_window)
        if np.sum(valid_mask) >= 21:
            y_v = y_window[valid_mask]
            x_v = x_window[valid_mask]
            #  Cov(x,y) / Var(x)
            cov_matrix = np.cov(y_v, x_v)
            if cov_matrix[1, 1] != 0:
                betas[i] = cov_matrix[0, 1] / cov_matrix[1, 1]
    
   
    df_group['beta'] = betas
    return df_group.loc[df_group['is_last_trade_day'] == 1, ['permno', 'date', 'beta']]


for col in ['exret', 'exmkf']:
    crsp[col] = crsp[col].astype('float32')


all_permnos = crsp['permno'].unique()
chunk_size = 300 
final_results = []


for i in range(0, len(all_permnos), chunk_size):
    subset_permnos = all_permnos[i:i+chunk_size]
    
    
    sub_df = crsp[crsp['permno'].isin(subset_permnos)].copy()
    
    
    res = sub_df.groupby('permno', group_keys=False).apply(calculate_beta_numpy)
    
    final_results.append(res.dropna(subset=['beta']))
    
    
    del sub_df, res
    gc.collect() 
    
    print(f"finished: {min(i+chunk_size, len(all_permnos))} / {len(all_permnos)}")

# 3. 合并最终结果
df_beta = pd.concat(final_results)
print("Success")




/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_1707/3015042777.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  res = sub_df.groupby('permno', group_keys=False).apply(calculate_beta_numpy)


finished: 300 / 13999


/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_1707/3015042777.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  res = sub_df.groupby('permno', group_keys=False).apply(calculate_beta_numpy)


finished: 600 / 13999


/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_1707/3015042777.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  res = sub_df.groupby('permno', group_keys=False).apply(calculate_beta_numpy)


...

/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_1707/3015042777.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  res = sub_df.groupby('permno', group_keys=False).apply(calculate_beta_numpy)


finished: 13500 / 13999


/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_1707/3015042777.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  res = sub_df.groupby('permno', group_keys=False).apply(calculate_beta_numpy)


finished: 13800 / 13999
finished: 13999 / 13999
Success


/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_1707/3015042777.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  res = sub_df.groupby('permno', group_keys=False).apply(calculate_beta_numpy)


In [ ]:
# df_beta_final=df_beta_final[df_beta_final['date']>='1957-01-31'] 
# df_beta_final = df_beta_final.reset_index(drop=True)
# df_beta_final.to_csv('/Users/ywo/Downloads/Industry Project/beta construction.csv', index=False)

In [7]:

df_beta['betasq']=df_beta['beta']**2
df_beta['beta_1']=df_beta.groupby('permno')['beta'].shift(1)
df_beta['betasq_1']=df_beta.groupby('permno')['betasq'].shift(1)

In [15]:
df_final=df_final.drop(columns=['beta', 'betasq'])
                              

In [16]:
df_beta['date'] = pd.to_datetime(df_beta['date'], errors='coerce')
merged=pd.merge(df_final,df_beta,on=['permno','date'],how='left')
merged 

,permno,date,ret,mvel1,retvol,chmom,agr,beta,betasq,beta_1,betasq_1
0,10006,1957-01-31,0.064378,82.249000,0.008058,0.047180,<NA>,0.876859,0.768881,0.912552,0.832751
1,10014,1957-01-31,0.095238,3.903375,0.033495,-0.275640,<NA>,-0.776824,0.603456,-0.225314,0.050767
2,10022,1957-01-31,0.102041,9.273250,0.015589,-0.025491,<NA>,0.034432,0.001186,0.036719,0.001348
3,10030,1957-01-31,-0.047091,54.465875,0.015849,0.018172,<NA>,0.707960,0.501207,0.663009,0.439581
4,10057,1957-01-31,-0.090062,40.250000,0.019945,0.025786,<NA>,1.052202,1.107129,0.742149,0.550784
...,...,...,...,...,...,...,...,...,...,...,...
1406401,93220,1987-12-31,0.274510,85.565250,0.037606,-0.964423,0.634408,0.181822,0.033059,0.163808,0.026833
1406402,93236,1987-12-31,-0.140000,9.493750,0.084282,-1.202767,-0.26887,1.044995,1.092014,1.123314,1.261835
1406403,93252,1987-12-31,-0.083333,9.630000,0.020063,-0.219780,-0.206293,0.352262,0.124089,0.427549,0.182798
1406404,93287,1987-12-31,NaN,21.858625,0.063171,NaN,-0.259728,NaN,NaN,NaN,NaN


In [17]:
merged=merged.drop(columns=['beta', 'betasq'])

,permno,date,ret,mvel1,retvol,chmom,agr,beta_1,betasq_1
0,10006,1957-01-31,0.064378,82.249000,0.008058,0.047180,<NA>,0.912552,0.832751
1,10014,1957-01-31,0.095238,3.903375,0.033495,-0.275640,<NA>,-0.225314,0.050767
2,10022,1957-01-31,0.102041,9.273250,0.015589,-0.025491,<NA>,0.036719,0.001348
3,10030,1957-01-31,-0.047091,54.465875,0.015849,0.018172,<NA>,0.663009,0.439581
4,10057,1957-01-31,-0.090062,40.250000,0.019945,0.025786,<NA>,0.742149,0.550784
...,...,...,...,...,...,...,...,...,...
1406401,93220,1987-12-31,0.274510,85.565250,0.037606,-0.964423,0.634408,0.163808,0.026833
1406402,93236,1987-12-31,-0.140000,9.493750,0.084282,-1.202767,-0.26887,1.123314,1.261835
1406403,93252,1987-12-31,-0.083333,9.630000,0.020063,-0.219780,-0.206293,0.427549,0.182798
1406404,93287,1987-12-31,NaN,21.858625,0.063171,NaN,-0.259728,NaN,NaN


In [19]:
merged.columns=['permno','date','ret','mvel1','retvol','chmom','agr','beta','betasq']
merged.to_csv('/Users/ywo/Downloads/Industry Project/feature construction_new.csv', index=False)

In [ ]:
# author's data
df2_beta=df2[['DATE','permno','beta']]
df2_beta=df2_beta[df2_beta['beta'].notna()]
df2_beta=df2_beta[df2_beta['permno']==10006]
df2_beta.head()

In [ ]:
df_beta=df1[['permno','date','ret','beta']]
df_beta['betasq']=df_beta['beta']**2
df_beta['beta_1']=df_beta.groupby('permno')['beta'].shift(1)
df_beta['betasq_1']=df_beta.groupby('permno')['betasq'].shift(1)
print(df_beta)

beta trend comparision

In [ ]:
# plt.figure(figsize=(10, 6))
# plt.plot(df_3_10014['date'], df_10014['beta'], label='Author Data', color='blue')
# plt.plot(df_3_10014['date'], df_3_10014['beta_1'], label='My Data', color='red')
# plt.xlabel('Date')
# plt.ylabel('Beta')
# plt.title('beta trend comparision')
# plt.legend() 
# plt.show()

Change in 6 month momentum

In [ ]:
df_del_mom=df2[(df2['DATE']>='1957-01-31') & (df2['DATE']<='1967-12-31')]
df_del_mom=df_del_mom[['permno','DATE','chmom','mom6m']]
df_del_mom_10006=df_del_mom[df_del_mom['permno']==10014]
print(df_del_mom_10006)

In [ ]:
query2= """
SELECT 
    permno, 
    date, 
    ret
    
FROM 
    crsp.msf 

WHERE 
    date >= '1956-01-31' AND date <= '1987-12-31'
    AND ret IS NOT NULL
"""
df_cal_mom = db.raw_sql(query2)

In [ ]:
df_cal_mom = df_cal_mom.drop_duplicates(subset=['permno', 'date'], keep='first')
df_cal_mom = df_cal_mom.reset_index(drop=True)
df_cal_mom['ret'] = pd.to_numeric(df_cal_mom['ret'], errors='coerce')
df_cal_mom['ret'] = df_cal_mom['ret'].astype('float64')

In [ ]:
# def calculate_mom(data):
#     mom_now_6=(1+data['ret']).shift(1).rolling(window=6).apply(np.prod, raw=True)-1
#     return mom_now_6
# df_cal_mom['mom6m']=df_cal_mom.groupby('permno', group_keys=False).apply(calculate_mom)

In [ ]:
def calculate_delta_mom(data):
    mom_now_6=(1+data['ret']).shift(1).rolling(window=6).apply(np.prod, raw=True)-1
    mom_pre_6=(1+data['ret']).shift(7).rolling(window=6).apply(np.prod, raw=True)-1
    return mom_now_6 - mom_pre_6
df_cal_mom['chmom']=df_cal_mom.groupby('permno', group_keys=False).apply(calculate_delta_mom)


In [ ]:
df_sd=df_cal_mom[(df_cal_mom['date']>='1957-01-31')]
df_sd = df_sd.reset_index(drop=True)
df_sd=df_sd[['permno','date','chmom']]
print(df_sd)

return volatility

In [ ]:
# author's data
df=df2[(df2['DATE']>='1956-10-31') & (df2['DATE']<='1987-12-31')]
df=df[['permno','DATE','retvol']]
print(df)

In [3]:
# calculate with wrds
query3= """
SELECT 
    permno, 
    date, 
    ret
    
FROM 
    crsp.dsf 

WHERE 
    date >= '1956-10-31' AND date <= '1987-12-31'
    AND ret IS NOT NULL
"""
df_cal_vol = db.raw_sql(query3)
df_cal_vol['date']=pd.to_datetime(df_cal_vol['date'])
df_cal_vol = df_cal_vol.drop_duplicates(subset=['permno', 'date'], keep='first')
df_cal_vol = df_cal_vol.reset_index(drop=True)
df_cal_vol['ret'] = pd.to_numeric(df_cal_vol['ret'], errors='coerce')
df_cal_vol['ret'] = df_cal_vol['ret'].astype('float64')
df_cal_vol['year_month'] = df_cal_vol['date'].dt.to_period('M')
# print(df_cal_vol)

In [4]:
print(df_cal_vol)

          permno       date       ret year_month
0          10006 1956-10-31 -0.021459    1956-10
1          10014 1956-10-31  0.000000    1956-10
2          10022 1956-10-31  0.000000    1956-10
3          10030 1956-10-31 -0.014793    1956-10
4          10057 1956-10-31 -0.013333    1956-10
...          ...        ...       ...        ...
28312118   93201 1987-12-31 -0.016854    1987-12
28312119   93220 1987-12-31  0.048387    1987-12
28312120   93236 1987-12-31  0.023810    1987-12
28312121   93252 1987-12-31 -0.029412    1987-12
28312122   93316 1987-12-31 -0.030303    1987-12

[28312123 rows x 4 columns]


In [5]:
df_cal_vol = df_cal_vol.sort_values(['permno', 'year_month'])
def calculate_volatility(data):
    if len(data) < 15:
        return np.nan
    return data['ret'].std()
df_cal_vol2 = df_cal_vol.groupby(['permno','year_month'], group_keys=False).apply(calculate_volatility).reset_index()


/var/folders/7z/3m4wyyc12cs4r040zrk52nf40000gn/T/ipykernel_6815/510702948.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cal_vol2 = df_cal_vol.groupby(['permno','year_month'], group_keys=False).apply(calculate_volatility).reset_index()


In [6]:
df_cal_vol2.columns=['permno','year_month','retvol']
df_cal_vol2['date'] = df_cal_vol2['year_month'].dt.to_timestamp('M')
print(df_cal_vol2)


         permno year_month    retvol       date
0         10000    1986-01  0.065278 1986-01-31
1         10000    1986-02  0.031004 1986-02-28
2         10000    1986-03  0.044548 1986-03-31
3         10000    1986-04  0.011246 1986-04-30
4         10000    1986-05  0.038862 1986-05-31
...         ...        ...       ...        ...
1361437   93324    1985-08  0.030457 1985-08-31
1361438   93324    1985-09  0.072503 1985-09-30
1361439   93324    1985-10  0.145448 1985-10-31
1361440   93324    1985-11  0.031944 1985-11-30
1361441   93324    1985-12       NaN 1985-12-31

[1361442 rows x 4 columns]


In [ ]:
df_vol=df_cal_vol2
df_vol['retvol_1']=df_vol.groupby('permno')['retvol'].shift(1)
print(df_vol)

In [ ]:
# merged1=pd.merge(df,df_cal_vol,left_on=['permno','DATE'],right_on=['permno','date'],how='right')
# merged1=merged1[['permno','date','year_month','ret']]
# print(merged1)
# merged2=pd.merge(merged1,df_cal_vol2,on=['permno','year_month'],how='left')
# print(merged2)

In [ ]:
# # final calculate volatility dataset
# df_vol=merged2[['permno','date','retvol']]
# df_vol=df_vol[(df_vol['date']>='1956-12-31')]
# df_vol = df_vol.reset_index(drop=True)
# df_vol['retvol_1']=df_vol.groupby('permno')['retvol'].shift(1)
# print(df_vol)


size

In [5]:
query = """
SELECT 
    permno,      
    date,        
     
    (abs(prc) * shrout) / 1000 AS mvel1 
FROM 
    crsp.msf
WHERE 
    date >= '1956-10-31' AND date <= '1987-12-31'
"""

In [6]:
df_size = db.raw_sql(query)
df_size = df_size.drop_duplicates(subset=['permno', 'date'], keep='first')
df_size = df_size.reset_index(drop=True)

In [7]:
df_size=df_size[df_size['date']>='1956-12-31']
df_size = df_size.reset_index(drop=True)
df_size['mvel1_1']=df_size.groupby('permno')['mvel1'].shift(1)
print(df_size[df_size['permno']==10006])

         permno        date      mvel1    mvel1_1
0         10006  1956-12-31     82.249       <NA>
1         10014  1956-12-31   3.903375       <NA>
2         10022  1956-12-31    9.27325       <NA>
3         10030  1956-12-31  54.465875       <NA>
4         10057  1956-12-31      40.25       <NA>
...         ...         ...        ...        ...
1407465   93220  1987-12-31  109.05375   85.56525
1407466   93236  1987-12-31   8.164625    9.49375
1407467   93252  1987-12-31     8.8275       9.63
1407468   93287  1987-12-31       <NA>  21.858625
1407469   93316  1987-12-31      5.316      5.316

[1407470 rows x 4 columns]


In [8]:
print(df_size[df_size['permno']==10006])

         permno        date       mvel1     mvel1_1
0         10006  1956-12-31      82.249        <NA>
1064      10006  1957-01-31      87.544      82.249
2130      10006  1957-02-28     86.3085      87.544
3195      10006  1957-03-29      87.897     86.3085
4264      10006  1957-04-30      87.191      87.897
...         ...         ...         ...         ...
1089719   10006  1984-02-29   400.38375  382.565625
1096185   10006  1984-03-30     427.635   400.38375
1102662   10006  1984-04-30  441.260625     427.635
1109165   10006  1984-05-31  441.260625  441.260625
1115701   10006  1984-06-29        <NA>  441.260625

[331 rows x 4 columns]


take beta, size, chmom, volatility together

In [ ]:
df_beta['date'] = pd.to_datetime(df_beta['date'])
df_size['date'] = pd.to_datetime(df_size['date'])
df_vol['date'] = pd.to_datetime(df_vol['date'])
df_sd['date'] = pd.to_datetime(df_sd['date'])
mergeda=pd.merge(df_beta,df_size,on=['permno','date'],how='right')
mergedb=pd.merge(mergeda,df_vol,on=['permno','date'],how='left')
mergedc=pd.merge(mergedb,df_sd,on=['permno','date'],how='left')
mergedc=mergedc[mergedc['date']>='1957-01-31']
mergedc = mergedc.reset_index(drop=True)
print(mergeda)
print(mergedb)
print(mergedc)

In [ ]:
df=mergedc[['permno','date','ret','beta_1','betasq_1','mvel1_1','retvol_1','chmom']]
df.columns=['permno','date','ret','beta','betasq','mvel1','retvol','chmom']
print(df)

In [ ]:
print(df[df['permno']==10006])

In [ ]:
df.to_csv('/Users/ywo/Downloads/Industry Project/feature construction.csv', index=False)


In [20]:
db.close()